In [ ]:

!pip install -q optuna

import gc,json,time,warnings
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold,train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.utils.class_weight import compute_sample_weight
warnings.filterwarnings("ignore")
ID="id";TARGET="health_condition";SEED=2027;FOLDS=5
C=np.array(["at-risk","fit","unhealthy"]);M={c:i for i,c in enumerate(C)}
base=Path("/kaggle/input/competitions/playground-series-s6e7");tr=pd.read_csv(base/"train.csv");te=pd.read_csv(base/"test.csv");y=tr[TARGET].map(M).to_numpy()
def fe(d):
 z=d.drop(columns=[ID,TARGET],errors="ignore").copy();s=z.sleep_duration
 z["sleep_lt6"]=np.where(s.isna(),np.nan,(s<6).astype(float));z["sleep_lt7"]=np.where(s.isna(),np.nan,(s<7).astype(float))
 z["key_missing_count"]=z[["sleep_duration","stress_level","physical_activity_level"]].isna().sum(1)
 z["key_pattern"]=z[["sleep_duration","stress_level","physical_activity_level"]].isna().astype(int).astype(str).agg("".join,axis=1)
 z["sleep_zone"]=pd.cut(s,[-np.inf,6,7,np.inf],right=False,labels=["lt6","6to7","ge7"]).astype("object")
 z["rule_unhealthy"]=((s<6)&z.stress_level.eq("high")).astype(float);z["rule_fit"]=((s>=7)&z.stress_level.eq("low")&z.physical_activity_level.eq("active")).astype(float)
 for c in z.select_dtypes(exclude=["object","category"]).columns:
  z[f"{c}_frac10"]=np.floor(z[c].abs()*10+1e-7)%10
 return z
X=fe(tr);T=fe(te);cats=list(X.select_dtypes(include=["object","category"]).columns);nums=[c for c in X if c not in cats]
e=OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1);e.fit(pd.concat([X[cats],T[cats]]).fillna("NA"));X[cats]=e.transform(X[cats].fillna("NA"));T[cats]=e.transform(T[cats].fillna("NA"))
med=X[nums].median();X[nums]=X[nums].fillna(med);T[nums]=T[nums].fillna(med);X=X.astype("float32");T=T.astype("float32")

import optuna
from sklearn.ensemble import HistGradientBoostingClassifier
rng=np.random.default_rng(SEED);pool=rng.choice(len(X),size=360000,replace=False);a,b=train_test_split(pool,test_size=.20,random_state=SEED,stratify=y[pool])
def objective(t):
 p=dict(learning_rate=t.suggest_float("learning_rate",.025,.14,log=True),max_iter=t.suggest_int("max_iter",250,900),
  max_leaf_nodes=t.suggest_int("max_leaf_nodes",15,127),max_depth=t.suggest_int("max_depth",4,14),
  min_samples_leaf=t.suggest_int("min_samples_leaf",15,100),l2_regularization=t.suggest_float("l2_regularization",1e-3,10,log=True),
  max_bins=t.suggest_int("max_bins",64,255),early_stopping=True,validation_fraction=.08,n_iter_no_change=35,random_state=SEED)
 m=HistGradientBoostingClassifier(**p);m.fit(X.iloc[a],y[a],sample_weight=compute_sample_weight("balanced",y[a]))
 return balanced_accuracy_score(y[b],m.predict_proba(X.iloc[b]).argmax(1))
study=optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler(seed=SEED));study.optimize(objective,n_trials=36)
best=study.best_params;print(study.best_value,best)
oof=np.zeros((len(X),3),np.float32);pred=np.zeros((len(T),3));fs=[];t0=time.time();skf=StratifiedKFold(FOLDS,shuffle=True,random_state=SEED)
for f,(a,b) in enumerate(skf.split(X,y),1):
 m=HistGradientBoostingClassifier(**best,early_stopping=True,validation_fraction=.08,n_iter_no_change=45,random_state=SEED+f)
 m.fit(X.iloc[a],y[a],sample_weight=compute_sample_weight("balanced",y[a]));oof[b]=m.predict_proba(X.iloc[b]);pred+=m.predict_proba(T)/FOLDS
 fs.append(float(balanced_accuracy_score(y[b],oof[b].argmax(1))));del m;gc.collect()
score=float(balanced_accuracy_score(y,oof.argmax(1)));od=pd.DataFrame({ID:tr[ID]});td=pd.DataFrame({ID:te[ID]})
for j,c in enumerate(C):od[c]=oof[:,j];td[c]=pred[:,j]
od.to_csv("oof_preds.csv",index=False);td.to_csv("test_preds.csv",index=False);pd.DataFrame({ID:te[ID],TARGET:C[pred.argmax(1)]}).to_csv("submission.csv",index=False)
Path("training_summary.json").write_text(json.dumps({"experiment":"optuna_hgb","tuning_best_score":study.best_value,"best_params":best,"fold_scores":fs,"oof_balanced_accuracy":score,"elapsed_minutes":(time.time()-t0)/60},indent=2));print(score)
